# 02 — ZoomMoE: model and staged training

Implements the three components of the proposal on top of the folds built in
notebook 01:

* **A — Progressive Zoom Refinement (PZR).** An ordered residual-gated state
  `h_k = h_{k-1} + g_k * delta_k`, where queries derived from `h_{k-1}` attend
  into the spatial tokens of magnification `m_k`. Replaces the permutation-
  invariant set attention of the baseline.
* **B — Zoom-conditioned Conv-MoE inside the CNN.** Sparse top-k expert blocks
  in the last two ResNet stages. The router sees the local feature, the
  magnification embedding, **and `h_{k-1}`** — so expert choice at 400x depends
  on what the coarser views already found. That feedback edge is the main
  architectural claim.
* **C — Contrastive alignment.** Cross-magnification InfoNCE with patient-safe
  negatives, an invariant/specific split with a gradient-reversal layer, a
  zoom-ordinal metric regulariser, and SupCon with cross-patient positives.

Training runs in three stages: unsupervised warm-up, supervised fusion,
end-to-end fine-tune.

**Cost warning.** The model runs the backbone once per magnification, so one
sample is four ResNet-50 forward passes. Start with `QUICK = True` to verify
the pipeline on a patient subset before committing to a full 5-fold run.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Add the folder containing breakhis_common.py and zoommoe_model.py
CODE_DIR = Path("/kaggle/input/datasets/nabinkoirala452/code-files-mc")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import breakhis_common as bc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
NUM_CLASSES = bc.N_CLASSES
CLASS_NAMES = list(bc.CLASS_NAMES)
OUT = Path("/kaggle/working/artifacts")
OUT.mkdir(parents=True, exist_ok=True)

bc.set_seed(SEED)
print("code directory:", CODE_DIR)
print("device:", DEVICE)
print("torch:", torch.__version__)
print("classes:", dict(enumerate(CLASS_NAMES)))

code directory: /kaggle/input/datasets/nabinkoirala452/code-files-mc
device: cuda
torch: 2.10.0+cu128
classes: {0: 'Adenosis', 1: 'Fibroadenoma', 2: 'Phyllodes Tumor', 3: 'Tubular Adenoma', 4: 'Ductal Carcinoma', 5: 'Lobular Carcinoma', 6: 'Mucinous Carcinoma', 7: 'Papillary Carcinoma'}


## 1. Model definition

Written to `zoommoe_model.py` so notebook 03 imports exactly the same code.

In [2]:
# The model is maintained in the shared code file so notebooks 02 and 03
# cannot silently diverge. Upload the updated zoommoe_model.py with the
# multiclass breakhis_common.py file to the Kaggle code-files dataset.
model_path = CODE_DIR / "zoommoe_model.py"
if not model_path.exists():
    raise FileNotFoundError(f"Missing shared model file: {model_path}")
print("using shared model:", model_path)

using shared model: /kaggle/input/datasets/nabinkoirala452/code-files-mc/zoommoe_model.py


In [3]:
import importlib
import zoommoe_model as zm

importlib.reload(zm)
print("model module loaded")

model module loaded


## 2. Smoke test on random tensors

Ten seconds, no dataset, no pretrained download. Run this first — it catches
every shape bug before you spend GPU hours.

In [4]:
def smoke_test():
    cfg = zm.ZoomMoEConfig(
        img_size=64, n_classes=NUM_CLASSES, n_experts=4, top_k=2,
        pretrained=False, state_dim=64, tok_dim=64, proj_dim=32, n_queries=4,
    )
    model = zm.ZoomMoE(cfg)
    B, M = 6, 4
    images = torch.randn(B, M, 3, 64, 64)
    mask = torch.ones(B, M, dtype=torch.bool)
    mask[0, 3] = False
    mask[1, 1] = False
    batch = {
        "images": images,
        "mask": mask,
        "label": torch.randint(0, NUM_CLASSES, (B,), dtype=torch.long),
        "patient_idx": torch.tensor([0, 0, 1, 1, 2, 3]),
    }

    out = model(images, mask)
    for k in ("logit", "state", "z_inv", "z_spec", "step_logits", "utility"):
        print(f"  {k:12s} {tuple(out[k].shape)}")

    assert out["logit"].shape == (B, NUM_CLASSES)
    assert out["step_logits"].shape == (B, M, NUM_CLASSES)
    total, parts = zm.compute_losses(out, batch, cfg)
    print("  loss terms:", {k: round(v, 4) for k, v in parts.items()})
    total.backward()

    grads = [p.grad is not None for p in model.parameters() if p.requires_grad]
    print(f"  parameters receiving gradient: {sum(grads)}/{len(grads)}")
    assert torch.isfinite(total), "loss is not finite"

    out_p = model(images, mask, order=[3, 2, 1, 0])
    delta = (out["state"] - out_p["state"]).abs().mean().item()
    print(f"  mean |state - state_permuted| = {delta:.5f}  (should be > 0)")

    usage = zm.expert_usage_table(out["route"], cfg.n_experts)
    print(f"  expert usage table {tuple(usage.shape)}, row sums {usage.sum(1).tolist()}")
    print("  params:", sum(p.numel() for p in model.parameters()))
    return model


_ = smoke_test()
del _
print("\nmulticlass smoke test passed")

  logit        (6, 8)
  state        (6, 64)
  z_inv        (6, 4, 32)
  z_spec       (6, 4, 32)
  step_logits  (6, 4, 8)
  utility      (6, 4)
  loss terms: {'cls': 1.968, 'deep': 0.6308, 'supcon': 0.7617, 'align': 1.6488, 'ordinal': 0.0521, 'disent': 0.0262, 'mag_adv': 0.2781, 'mag_spec': 0.2782, 'bal': 0.0204, 'moe_spec': -0.0001, 'zloss': 0.0025}
  parameters receiving gradient: 237/237
  mean |state - state_permuted| = 0.03083  (should be > 0)
  expert usage table (4, 4), row sums [1.0, 1.0, 1.0, 1.0]
  params: 36367584

multiclass smoke test passed


## 3. Load folds from notebook 01

In [5]:
samples = pd.read_pickle("/kaggle/input/datasets/nabinkoirala452/articafts-notebook1-mc/artifacts/samples_pool.pkl")   # pooled pairing = free augmentation
samples_ref = pd.read_pickle("/kaggle/input/datasets/nabinkoirala452/articafts-notebook1-mc/artifacts/samples_all.pkl")
folds = list(np.load("/kaggle/input/datasets/nabinkoirala452/articafts-notebook1-mc/artifacts/folds_all.npy", allow_pickle=True))

# folds were built on samples_all; the two tables must be row-aligned
assert (samples.sample_id.values == samples_ref.sample_id.values).all(), \
    "sample tables are not aligned - rebuild both in notebook 01"
bc.audit_folds(samples, folds)

QUICK = False         # subsample patients for a fast pipeline check
N_FOLDS = 1 if QUICK else 5

if QUICK:
    keep = samples.patient_id.isin(sorted(samples.patient_id.unique())[:20])
    print(f"QUICK mode: {int(keep.sum())} of {len(samples)} samples, 20 patients")

fold 0: n={'train': 1559, 'val': 324, 'test': 524}  patients={'train': 56, 'val': 10, 'test': 16}  class counts={'train': [88, 226, 17, 70, 777, 146, 190, 45], 'val': [32, 32, 42, 37, 47, 0, 42, 92], 'test': [0, 60, 66, 56, 248, 55, 22, 17]}
fold 1: n={'train': 1616, 'val': 287, 'test': 504}  patients={'train': 56, 'val': 10, 'test': 16}  class counts={'train': [88, 230, 59, 148, 725, 146, 175, 45], 'val': [32, 32, 66, 15, 49, 0, 22, 71], 'test': [0, 56, 0, 0, 298, 55, 57, 38]}
fold 2: n={'train': 1703, 'val': 228, 'test': 476}  patients={'train': 55, 'val': 10, 'test': 17}  class counts={'train': [52, 200, 66, 110, 839, 179, 184, 73], 'val': [36, 37, 17, 16, 37, 22, 27, 36], 'test': [32, 81, 42, 37, 196, 0, 43, 45]}
fold 3: n={'train': 1625, 'val': 302, 'test': 480}  patients={'train': 56, 'val': 10, 'test': 16}  class counts={'train': [17, 179, 108, 95, 824, 145, 212, 45], 'val': [32, 18, 0, 37, 99, 0, 42, 74], 'test': [71, 121, 17, 31, 149, 56, 0, 35]}
fold 4: n={'train': 1701, 'val

## 4. Staged training

| stage | trains | losses | why |
|---|---|---|---|
| 0 | backbone + MoE + projection heads | align, ordinal, disentangle, routing | experts differentiate before any class signal can bias them; uses all images, no labels |
| 1 | PZR + heads, early backbone frozen | cls, deep, supcon (+ small unsupervised) | fit the classifier on stable features |
| 2 | everything, low LR | all | joint refinement, early-exit calibration |

In [6]:
from contextlib import nullcontext

from torch.utils.data import DataLoader


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_ctx(enabled):
    if not enabled:
        return nullcontext()
    try:
        return torch.amp.autocast("cuda")
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast()


def make_loader(idx, train, batch_size, scale_dropout=0.0, img_size=224, workers=2):
    ds = bc.make_image_dataset(
        samples.iloc[idx], train=train, img_size=img_size,
        scale_dropout=scale_dropout, seed=SEED,
    )
    return DataLoader(
        ds, batch_size=batch_size, shuffle=train, num_workers=workers,
        pin_memory=True, drop_last=train,
    )


def run_stage(model, loader, cfg, epochs, lr, supervised, unsupervised,
              freeze=None, grl_ramp=True, amp=True, log_every=20, tag=""):
    model.backbone.freeze_until(freeze)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=max(epochs * len(loader), 1), pct_start=0.3
    )
    use_amp = amp and DEVICE.type == "cuda"
    scaler = make_scaler(use_amp)

    for ep in range(epochs):
        model.train()
        running = []
        for it, batch in enumerate(loader):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            frac = (ep * len(loader) + it) / max(epochs * len(loader), 1)
            lam = float(2.0 / (1.0 + np.exp(-10 * frac)) - 1.0) if grl_ramp else 1.0

            with autocast_ctx(use_amp):
                out = model(batch["images"], batch["mask"], grl_lambda=lam)
                total, parts = zm.compute_losses(
                    out, batch, cfg, supervised=supervised, unsupervised=unsupervised
                )

            opt.zero_grad(set_to_none=True)
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 5.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            running.append(parts)

            if it % log_every == 0:
                msg = "  ".join(f"{k}={v:.3f}" for k, v in parts.items())
                print(f"  [{tag}] ep{ep} it{it}: total={float(total):.3f}  {msg}", flush=True)

        if not running:
            print(f"  [{tag}] epoch {ep}: empty loader, nothing to train on")
            continue
        mean = {k: float(np.mean([r[k] for r in running])) for k in running[0]}
        print(f"  [{tag}] epoch {ep} mean: " + "  ".join(f"{k}={v:.3f}" for k, v in mean.items()))
    return model


@torch.no_grad()
def evaluate_loader(model, loader):
    res = zm.collect(model, loader, DEVICE)
    m = bc.evaluate(res["label"], res["prob"], res["patient_idx"])
    return m, res

In [7]:
import sys

# 1. The read-only path we want Python to STOP looking at
bad_path = "/kaggle/input/datasets/nabinkoirala452/code-files_mc"

# 2. Remove it from sys.path so Python can't find the old file
while bad_path in sys.path:
    sys.path.remove(bad_path)

# 3. Add the working directory (where our fixed file is) to the front
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# 4. Brutally delete the cached module from Python's memory
if 'zoommoe_model' in sys.modules:
    del sys.modules['zoommoe_model']

# 5. Re-import it. It has no choice but to grab the new one now!
import zoommoe_model as zm

print("Now using the model file from:")
print(zm.__file__)

Now using the model file from:
/kaggle/input/datasets/nabinkoirala452/code-files-mc/zoommoe_model.py


In [8]:
BATCH = 8               # 4 backbone passes per sample; raise if memory allows
EPOCHS_S0, EPOCHS_S1, EPOCHS_S2 = (1, 2, 1) if QUICK else (8, 15, 8)
IMG = 224

cfg = zm.ZoomMoEConfig(img_size=IMG, n_classes=NUM_CLASSES, n_experts=8, top_k=2, pretrained=True)
fold_results, histories = [], {}

for fi in range(N_FOLDS):
    fold = folds[fi]
    if QUICK:
        fold = {k: np.array([i for i in v if keep.iloc[i]]) for k, v in fold.items()}
    print(f"\n================ fold {fi} ================")
    print({k: len(v) for k, v in fold.items()})
    bc.set_seed(SEED + fi)

    model = zm.ZoomMoE(cfg).to(DEVICE)
    print("trainable params:", bc.count_params(model))

    ssl_loader = make_loader(fold["train"], True, BATCH, scale_dropout=0.15, img_size=IMG)
    sup_loader = make_loader(fold["train"], True, BATCH, scale_dropout=0.15, img_size=IMG)
    val_loader = make_loader(fold["val"], False, BATCH, img_size=IMG)
    test_loader = make_loader(fold["test"], False, BATCH, img_size=IMG)

    # stage 0 - unsupervised warm-up
    run_stage(model, ssl_loader, cfg, EPOCHS_S0, 3e-4,
              supervised=False, unsupervised=True, freeze="layer2", tag=f"f{fi}-s0")

    # stage 1 - supervised eight-class fusion
    run_stage(model, sup_loader, cfg, EPOCHS_S1, 3e-4,
              supervised=True, unsupervised=True, freeze="layer2", tag=f"f{fi}-s1")
    vm, _ = evaluate_loader(model, val_loader)
    print(f"  [f{fi}] after stage 1, val:", {k: round(v, 4) for k, v in vm.items()})

    # stage 2 - end-to-end fine-tune
    run_stage(model, sup_loader, cfg, EPOCHS_S2, 3e-5,
              supervised=True, unsupervised=True, freeze=None, tag=f"f{fi}-s2")

    vm, _ = evaluate_loader(model, val_loader)
    tm, test_res = evaluate_loader(model, test_loader)
    print(f"  [f{fi}] val :", {k: round(v, 4) for k, v in vm.items()})
    print(f"  [f{fi}] TEST:", {k: round(v, 4) for k, v in tm.items()})

    tm["fold"] = fi
    fold_results.append(tm)
    torch.save({"cfg": cfg, "state_dict": model.state_dict()}, OUT / f"zoommoe_multiclass_fold{fi}.pt")
    np.savez_compressed(OUT / f"zoommoe_multiclass_test_fold{fi}.npz", **test_res)
    del model
    torch.cuda.empty_cache()


================ fold 0 ================
{'train': 1559, 'val': 324, 'test': 524}
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s]


trainable params: 48542504
  [f0-s0] ep0 it0: total=2.048  align=1.420  ordinal=0.020  disent=0.024  mag_adv=0.277  mag_spec=0.279  bal=0.024  moe_spec=-0.001  zloss=0.005


/tmp/ipykernel_24/3117464509.py:69: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"  [{tag}] ep{ep} it{it}: total={float(total):.3f}  {msg}", flush=True)


  [f0-s0] ep0 it20: total=2.219  align=1.584  ordinal=0.031  disent=0.022  mag_adv=0.278  mag_spec=0.277  bal=0.023  moe_spec=-0.000  zloss=0.005
  [f0-s0] ep0 it40: total=1.783  align=1.145  ordinal=0.034  disent=0.024  mag_adv=0.278  mag_spec=0.275  bal=0.023  moe_spec=-0.001  zloss=0.005
  [f0-s0] ep0 it60: total=2.530  align=1.911  ordinal=0.015  disent=0.021  mag_adv=0.277  mag_spec=0.279  bal=0.023  moe_spec=-0.001  zloss=0.005
  [f0-s0] ep0 it80: total=2.126  align=1.494  ordinal=0.029  disent=0.023  mag_adv=0.278  mag_spec=0.276  bal=0.022  moe_spec=-0.001  zloss=0.004
  [f0-s0] ep0 it100: total=1.792  align=1.167  ordinal=0.018  disent=0.025  mag_adv=0.277  mag_spec=0.279  bal=0.023  moe_spec=-0.001  zloss=0.004
  [f0-s0] ep0 it120: total=1.376  align=0.728  ordinal=0.039  disent=0.027  mag_adv=0.279  mag_spec=0.277  bal=0.023  moe_spec=-0.001  zloss=0.004
  [f0-s0] ep0 it140: total=1.436  align=0.795  ordinal=0.036  disent=0.023  mag_adv=0.277  mag_spec=0.278  bal=0.024  moe_

...

  [f3-s1] ep0 it20: total=5.775  cls=2.214  deep=1.177  supcon=2.043  align=0.014  ordinal=0.000  disent=0.005  mag_adv=0.279  mag_spec=0.018  bal=0.046  moe_spec=-0.023  zloss=0.003
  [f3-s1] ep0 it40: total=5.858  cls=2.086  deep=1.056  supcon=2.358  align=0.031  ordinal=0.000  disent=0.005  mag_adv=0.277  mag_spec=0.019  bal=0.043  moe_spec=-0.020  zloss=0.003
  [f3-s1] ep0 it60: total=4.892  cls=1.709  deep=0.694  supcon=2.147  align=0.014  ordinal=0.000  disent=0.007  mag_adv=0.277  mag_spec=0.018  bal=0.045  moe_spec=-0.022  zloss=0.003
  [f3-s1] ep0 it80: total=5.585  cls=2.158  deep=0.934  supcon=2.098  align=0.067  ordinal=0.000  disent=0.003  mag_adv=0.277  mag_spec=0.020  bal=0.045  moe_spec=-0.020  zloss=0.003
  [f3-s1] ep0 it100: total=4.166  cls=1.647  deep=0.646  supcon=1.506  align=0.039  ordinal=0.000  disent=0.006  mag_adv=0.276  mag_spec=0.018  bal=0.046  moe_spec=-0.022  zloss=0.003
  [f3-s1] ep0 it120: total=4.292  cls=1.720  deep=0.405  supcon=1.755  align=0.083  

/tmp/ipykernel_24/3117464509.py:64: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  [f4-s1] ep0 it20: total=6.340  cls=2.702  deep=0.852  supcon=2.441  align=0.024  ordinal=0.000  disent=0.004  mag_adv=0.277  mag_spec=0.016  bal=0.048  moe_spec=-0.026  zloss=0.003
  [f4-s1] ep0 it40: total=6.285  cls=2.798  deep=0.710  supcon=2.436  align=0.016  ordinal=0.000  disent=0.004  mag_adv=0.279  mag_spec=0.016  bal=0.049  moe_spec=-0.025  zloss=0.003
  [f4-s1] ep0 it60: total=4.495  cls=2.148  deep=0.645  supcon=1.355  align=0.025  ordinal=0.000  disent=0.002  mag_adv=0.278  mag_spec=0.016  bal=0.049  moe_spec=-0.026  zloss=0.003
  [f4-s1] ep0 it80: total=4.949  cls=2.141  deep=0.647  supcon=1.834  align=0.004  ordinal=0.000  disent=0.005  mag_adv=0.277  mag_spec=0.016  bal=0.050  moe_spec=-0.027  zloss=0.003
  [f4-s1] ep0 it100: total=4.014  cls=2.186  deep=0.618  supcon=0.873  align=0.008  ordinal=0.004  disent=0.005  mag_adv=0.276  mag_spec=0.016  bal=0.051  moe_spec=-0.026  zloss=0.003
  [f4-s1] ep0 it120: total=4.874  cls=2.210  deep=0.399  supcon=1.545  align=0.398  

/tmp/ipykernel_24/3117464509.py:64: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  [f4-s2] ep0 it20: total=1.000  cls=0.025  deep=0.004  supcon=0.645  align=0.021  ordinal=0.000  disent=0.002  mag_adv=0.278  mag_spec=0.001  bal=0.050  moe_spec=-0.029  zloss=0.002
  [f4-s2] ep0 it40: total=0.977  cls=0.013  deep=0.004  supcon=0.642  align=0.013  ordinal=0.000  disent=0.003  mag_adv=0.278  mag_spec=0.001  bal=0.051  moe_spec=-0.029  zloss=0.002
  [f4-s2] ep0 it60: total=1.106  cls=0.217  deep=0.095  supcon=0.422  align=0.066  ordinal=0.000  disent=0.002  mag_adv=0.278  mag_spec=0.001  bal=0.050  moe_spec=-0.028  zloss=0.002
  [f4-s2] ep0 it80: total=1.681  cls=0.575  deep=0.128  supcon=0.641  align=0.025  ordinal=0.000  disent=0.008  mag_adv=0.278  mag_spec=0.001  bal=0.052  moe_spec=-0.029  zloss=0.002
  [f4-s2] ep0 it100: total=1.284  cls=0.006  deep=0.003  supcon=0.809  align=0.158  ordinal=0.000  disent=0.004  mag_adv=0.277  mag_spec=0.001  bal=0.054  moe_spec=-0.029  zloss=0.002
  [f4-s2] ep0 it120: total=1.163  cls=0.035  deep=0.025  supcon=0.773  align=0.022  

In [9]:
summary = bc.summarise_folds(fold_results)
display(summary.round(4))
pd.DataFrame(fold_results).to_csv(OUT / "zoommoe_multiclass_results.csv", index=False)

try:
    base = pd.read_csv(OUT / "baseline_results_honest_multiclass.csv", index_col=0)
    print("\nmulticlass baselines from notebook 01:")
    display(base)
except FileNotFoundError:
    print("run notebook 01 first for the multiclass baseline comparison")

,mean,std
acc,0.4959,0.1188
f1_macro,0.2612,0.0751
prr,0.5319,0.1035
auc,0.7399,0.0931
fold,2.0000,1.5811


run notebook 01 first for the multiclass baseline comparison


## 5. Ablation driver

One switch per claim. Each row removes exactly one thing, so a drop is
attributable. Run with `QUICK = False` and a single fold first; the full grid
across five folds is expensive.

In [10]:
ABLATIONS = {
    "full": {},
    "no_moe": dict(moe_stages=()),
    "moe_mag_only": dict(cond_state_dim=0),        # router loses h_{k-1}
    "no_contrastive": dict(w_align=0.0, w_ordinal=0.0, w_disent=0.0, w_advmag=0.0, w_supcon=0.0),
    "no_ordinal": dict(w_ordinal=0.0),
    "no_supcon": dict(w_supcon=0.0),
    "no_moe_spec_loss": dict(w_spec=0.0),
    "experts_4": dict(n_experts=4),
    "experts_16": dict(n_experts=16),
    "topk_1": dict(top_k=1),
}


def run_ablation(name, overrides, fold, epochs=(2, 4, 2), batch=BATCH):
    from dataclasses import replace

    acfg = replace(cfg, **overrides)
    bc.set_seed(SEED)
    model = zm.ZoomMoE(acfg).to(DEVICE)
    tr = make_loader(fold["train"], True, batch, scale_dropout=0.15, img_size=acfg.img_size)
    te = make_loader(fold["test"], False, batch, img_size=acfg.img_size)
    run_stage(model, tr, acfg, epochs[0], 3e-4, False, True, freeze="layer2", tag=f"{name}-s0")
    run_stage(model, tr, acfg, epochs[1], 3e-4, True, True, freeze="layer2", tag=f"{name}-s1")
    run_stage(model, tr, acfg, epochs[2], 3e-5, True, True, freeze=None, tag=f"{name}-s2")
    m, _ = evaluate_loader(model, te)
    m["ablation"] = name
    m["params"] = bc.count_params(model)
    del model
    torch.cuda.empty_cache()
    return m


RUN_ABLATIONS = False   # flip on once the full model trains cleanly

if RUN_ABLATIONS:
    fold = folds[0]
    if QUICK:
        fold = {k: np.array([i for i in v if keep.iloc[i]]) for k, v in fold.items()}
    rows = [run_ablation(n, o, fold) for n, o in ABLATIONS.items()]
    ab = pd.DataFrame(rows).set_index("ablation")
    display(ab.round(4))
    ab.to_csv(OUT / "ablations.csv")

### Note on `moe_mag_only`

`cond_state_dim=0` makes `state_to_cond` emit a zero-width vector, so the
router sees only the local feature and the magnification embedding. That is
the standard conditional-MoE setup, and the gap between it and `full` isolates
the contribution of the zoom-state feedback edge — the single most important
number in the ablation table.

## 6. Handoff

Written to `artifacts/`: `zoommoe_multiclass_fold*.pt`,
`zoommoe_multiclass_test_fold*.npz`, `zoommoe_multiclass_results.csv`, and
shared `zoommoe_model.py`.

The classifier predicts the eight BreaKHis subtypes defined in
`breakhis_common.CLASS_NAMES`. Next: `03_analysis_and_falsification.ipynb`,
which runs the tests that could show this whole direction does nothing.